## Celda 0 — Configuración (Drive, rutas y Gradio)

In [1]:
%pip install -q gradio

In [2]:
from google.colab import drive
import os

# Montar Google Drive
drive.mount('/content/drive')

# Ruta base del proyecto en Drive
RUTA_BASE = '/content/drive/MyDrive/SGB/datos/'
os.makedirs(RUTA_BASE, exist_ok=True)

# Rutas de cada archivo de persistencia
RUTA_USUARIOS     = RUTA_BASE + 'usuarios.txt'
RUTA_LIBROS       = RUTA_BASE + 'libros.txt'
RUTA_CATEGORIAS   = RUTA_BASE + 'categorias.txt'
RUTA_PRESTAMOS    = RUTA_BASE + 'prestamos.txt'
RUTA_MULTAS       = RUTA_BASE + 'multas.txt'
RUTA_RESERVAS     = RUTA_BASE + 'reservas.txt'

Mounted at /content/drive


# SECCIÓN 1 — MODELO (dominio)
## Celda 1.1 — Enums y utilidades comunes

In [3]:
from enum import Enum
from datetime import datetime, date, timedelta

SEP = "|"                    # delimitador de campos en los archivos .txt
FORMATO_FECHA = "%Y-%m-%d"   # formato de fechas: AAAA-MM-DD


def _limpiar(texto):
    """Elimina del texto el delimitador y los saltos de línea para no corromper el archivo."""
    return str(texto).replace(SEP, "/").replace("\n", " ").strip()


def parse_fecha(texto):
    """Convierte un texto AAAA-MM-DD en un objeto date. Lanza ValueError si es inválido."""
    try:
        return datetime.strptime(str(texto).strip(), FORMATO_FECHA).date()
    except ValueError:
        raise ValueError(f"Fecha inválida '{texto}'. Use el formato AAAA-MM-DD (ej. 2026-10-26).")


def fmt_fecha(f):
    """Convierte un objeto date en texto AAAA-MM-DD."""
    return f.strftime(FORMATO_FECHA)


class TipoUsuarioEnum(Enum):
    """Tipos de usuario con sus reglas de préstamo (días de plazo y cupo máximo)."""
    ESTUDIANTE = "ESTUDIANTE"
    DOCENTE = "DOCENTE"
    EXTERNO = "EXTERNO"

    @property
    def dias_prestamo(self):
        """Días de plazo del préstamo según el tipo de usuario."""
        return {"ESTUDIANTE": 7, "DOCENTE": 15, "EXTERNO": 5}[self.value]

    @property
    def cupo(self):
        """Cantidad máxima de libros simultáneos según el tipo de usuario."""
        return {"ESTUDIANTE": 3, "DOCENTE": 5, "EXTERNO": 2}[self.value]


class EstadoPrestamoEnum(Enum):
    """Estados posibles de un préstamo."""
    ACTIVO = "ACTIVO"
    DEVUELTO = "DEVUELTO"
    VENCIDO = "VENCIDO"


class EstadoMultaEnum(Enum):
    """Estados posibles de una multa."""
    PENDIENTE = "PENDIENTE"
    PAGADA = "PAGADA"


class EstadoReservaEnum(Enum):
    """Estados posibles de una reserva."""
    ACTIVA = "ACTIVA"
    CUMPLIDA = "CUMPLIDA"
    CANCELADA = "CANCELADA"

## Celda 1.2 — Clase `Usuario`

In [4]:
class Usuario:
    """Persona registrada en la biblioteca (estudiante, docente o externo)."""
    ACTIVO = "ACTIVO"
    SUSPENDIDO = "SUSPENDIDO"

    def __init__(self, num_documento, nombre, tipo, correo, telefono, estado="ACTIVO"):
        """Crea un usuario. `tipo` es un TipoUsuarioEnum; `estado` es ACTIVO o SUSPENDIDO."""
        self.num_documento = _limpiar(num_documento)
        self.nombre = _limpiar(nombre)
        self.tipo = tipo
        self.correo = _limpiar(correo)
        self.telefono = _limpiar(telefono)
        self.estado = estado

    def esta_activo(self):
        """Indica si el usuario está habilitado para prestar."""
        return self.estado == Usuario.ACTIVO

    def to_linea(self):
        """Serializa el usuario como una línea de texto delimitada."""
        return SEP.join([self.num_documento, self.nombre, self.tipo.value,
                         self.correo, self.telefono, self.estado])

    @staticmethod
    def from_linea(linea):
        """Reconstruye un Usuario a partir de una línea de texto."""
        p = linea.split(SEP)
        return Usuario(p[0], p[1], TipoUsuarioEnum(p[2]), p[3], p[4], p[5])

## Celda 1.3 — Clase `Categoria`

In [5]:
class Categoria:
    """Categoría temática que agrupa libros."""

    def __init__(self, codigo, nombre, descripcion=""):
        """Crea una categoría con código único, nombre y descripción."""
        self.codigo = _limpiar(codigo).upper()
        self.nombre = _limpiar(nombre)
        self.descripcion = _limpiar(descripcion)

    def to_linea(self):
        """Serializa la categoría como una línea de texto delimitada."""
        return SEP.join([self.codigo, self.nombre, self.descripcion])

    @staticmethod
    def from_linea(linea):
        """Reconstruye una Categoria a partir de una línea de texto."""
        p = linea.split(SEP)
        return Categoria(p[0], p[1], p[2])

## Celda 1.4 — Clase `Libro`

In [6]:
class Libro:
    """Libro del catálogo con control de ejemplares totales y disponibles."""

    def __init__(self, isbn, titulo, autor, cod_categoria, anio, total_ejemplares, disponibles=None):
        """Crea un libro. Si no se indica `disponibles`, se asume igual al total."""
        self.isbn = _limpiar(isbn)
        self.titulo = _limpiar(titulo)
        self.autor = _limpiar(autor)
        self.cod_categoria = _limpiar(cod_categoria).upper()
        self.anio = int(anio)
        self.total_ejemplares = int(total_ejemplares)
        self.disponibles = self.total_ejemplares if disponibles is None else int(disponibles)

    def to_linea(self):
        """Serializa el libro como una línea de texto delimitada."""
        return SEP.join([self.isbn, self.titulo, self.autor, self.cod_categoria,
                         str(self.anio), str(self.total_ejemplares), str(self.disponibles)])

    @staticmethod
    def from_linea(linea):
        """Reconstruye un Libro a partir de una línea de texto."""
        p = linea.split(SEP)
        return Libro(p[0], p[1], p[2], p[3], int(p[4]), int(p[5]), int(p[6]))

## Celda 1.5 — Clase `Prestamo`

In [7]:
class Prestamo:
    """Préstamo de un libro a un usuario."""

    def __init__(self, codigo, isbn, num_documento, fecha_prestamo, fecha_limite,
                 fecha_devolucion="", estado=EstadoPrestamoEnum.ACTIVO):
        """Crea un préstamo. `fecha_devolucion` queda vacía mientras no se devuelva."""
        self.codigo = codigo
        self.isbn = isbn
        self.num_documento = num_documento
        self.fecha_prestamo = fecha_prestamo
        self.fecha_limite = fecha_limite
        self.fecha_devolucion = fecha_devolucion
        self.estado = estado

    def esta_pendiente(self):
        """True si el libro aún no ha sido devuelto."""
        return self.fecha_devolucion == ""

    def to_linea(self):
        """Serializa el préstamo como una línea de texto delimitada."""
        return SEP.join([self.codigo, self.isbn, self.num_documento, self.fecha_prestamo,
                         self.fecha_limite, self.fecha_devolucion, self.estado.value])

    @staticmethod
    def from_linea(linea):
        """Reconstruye un Prestamo a partir de una línea de texto."""
        p = linea.split(SEP)
        return Prestamo(p[0], p[1], p[2], p[3], p[4], p[5], EstadoPrestamoEnum(p[6]))

## Celda 1.6 — Clase `Multa`

In [8]:
class Multa:
    """Multa generada por devolución tardía de un préstamo."""

    def __init__(self, codigo, cod_prestamo, dias_retraso, valor, estado=EstadoMultaEnum.PENDIENTE):
        """Crea una multa asociada a un préstamo."""
        self.codigo = codigo
        self.cod_prestamo = cod_prestamo
        self.dias_retraso = int(dias_retraso)
        self.valor = int(valor)
        self.estado = estado

    def to_linea(self):
        """Serializa la multa como una línea de texto delimitada."""
        return SEP.join([self.codigo, self.cod_prestamo, str(self.dias_retraso),
                         str(self.valor), self.estado.value])

    @staticmethod
    def from_linea(linea):
        """Reconstruye una Multa a partir de una línea de texto."""
        p = linea.split(SEP)
        return Multa(p[0], p[1], int(p[2]), int(p[3]), EstadoMultaEnum(p[4]))

## Celda 1.7 — Clase `Reserva`

In [9]:
class Reserva:
    """Reserva de un libro sin ejemplares disponibles (se atiende por orden de llegada)."""

    def __init__(self, codigo, isbn, num_documento, fecha_reserva, estado=EstadoReservaEnum.ACTIVA):
        """Crea una reserva de un usuario sobre un libro."""
        self.codigo = codigo
        self.isbn = isbn
        self.num_documento = num_documento
        self.fecha_reserva = fecha_reserva
        self.estado = estado

    def to_linea(self):
        """Serializa la reserva como una línea de texto delimitada."""
        return SEP.join([self.codigo, self.isbn, self.num_documento,
                         self.fecha_reserva, self.estado.value])

    @staticmethod
    def from_linea(linea):
        """Reconstruye una Reserva a partir de una línea de texto."""
        p = linea.split(SEP)
        return Reserva(p[0], p[1], p[2], p[3], EstadoReservaEnum(p[4]))

# SECCIÓN 2 — REPOSITORIOS (persistencia en archivos de texto)
## Celda 2.1 — `ArchivoUtil` y repositorio base

In [10]:
import os
import uuid


class ArchivoUtil:
    """Utilidades de bajo nivel para leer y escribir archivos de texto."""

    @staticmethod
    def leer_lineas(ruta):
        """Devuelve la lista de líneas no vacías del archivo (lista vacía si no existe)."""
        if not os.path.exists(ruta):
            return []
        with open(ruta, "r", encoding="utf-8") as f:
            return [l.strip("\n") for l in f if l.strip()]

    @staticmethod
    def escribir_linea(ruta, linea):
        """Agrega una línea al final del archivo."""
        with open(ruta, "a", encoding="utf-8") as f:
            f.write(linea + "\n")

    @staticmethod
    def reescribir(ruta, lineas):
        """Reemplaza todo el contenido del archivo por las líneas dadas."""
        with open(ruta, "w", encoding="utf-8") as f:
            for l in lineas:
                f.write(l + "\n")

    @staticmethod
    def generar_codigo(prefijo):
        """Genera un código único con el prefijo dado (ej. PRE-3FA9C1)."""
        return f"{prefijo}-{uuid.uuid4().hex[:6].upper()}"


class RepositorioBase:
    """Comportamiento común de los repositorios: leer todo, guardar y reemplazar por clave."""

    def __init__(self, ruta, clase):
        """Recibe la ruta del archivo y la clase del modelo que persiste."""
        self.ruta = ruta
        self.clase = clase

    def find_all(self):
        """Devuelve todos los registros del archivo como objetos del modelo."""
        return [self.clase.from_linea(l) for l in ArchivoUtil.leer_lineas(self.ruta)]

    def guardar(self, obj):
        """Agrega un nuevo registro al archivo."""
        ArchivoUtil.escribir_linea(self.ruta, obj.to_linea())

    def _reemplazar(self, obj, atributo_clave):
        """Reemplaza el registro cuya clave coincide con la del objeto dado."""
        items = self.find_all()
        for i, x in enumerate(items):
            if getattr(x, atributo_clave) == getattr(obj, atributo_clave):
                items[i] = obj
                break
        ArchivoUtil.reescribir(self.ruta, [x.to_linea() for x in items])

## Celda 2.2 — `UsuarioRepository`

In [11]:
class UsuarioRepository(RepositorioBase):
    """Persistencia de usuarios en usuarios.txt."""

    def __init__(self, ruta=None):
        """Usa por defecto RUTA_USUARIOS."""
        super().__init__(ruta or RUTA_USUARIOS, Usuario)

    def find_by_documento(self, doc):
        """Busca un usuario por número de documento; None si no existe."""
        return next((u for u in self.find_all() if u.num_documento == doc.strip()), None)

    def actualizar(self, u):
        """Actualiza los datos de un usuario existente."""
        self._reemplazar(u, "num_documento")

## Celda 2.3 — `CategoriaRepository`

In [12]:
class CategoriaRepository(RepositorioBase):
    """Persistencia de categorías en categorias.txt."""

    def __init__(self, ruta=None):
        """Usa por defecto RUTA_CATEGORIAS."""
        super().__init__(ruta or RUTA_CATEGORIAS, Categoria)

    def find_by_codigo(self, cod):
        """Busca una categoría por código; None si no existe."""
        return next((c for c in self.find_all() if c.codigo == cod.strip().upper()), None)

## Celda 2.4 — `LibroRepository`

In [13]:
class LibroRepository(RepositorioBase):
    """Persistencia de libros en libros.txt."""

    def __init__(self, ruta=None):
        """Usa por defecto RUTA_LIBROS."""
        super().__init__(ruta or RUTA_LIBROS, Libro)

    def find_by_isbn(self, isbn):
        """Busca un libro por ISBN; None si no existe."""
        return next((l for l in self.find_all() if l.isbn == isbn.strip()), None)

    def find_by_titulo(self, txt):
        """Libros cuyo título contiene el texto (sin distinguir mayúsculas)."""
        t = txt.strip().lower()
        return [l for l in self.find_all() if t in l.titulo.lower()]

    def find_by_categoria(self, cod):
        """Libros de una categoría."""
        return [l for l in self.find_all() if l.cod_categoria == cod.strip().upper()]

    def actualizar(self, l):
        """Actualiza los datos de un libro existente."""
        self._reemplazar(l, "isbn")

## Celda 2.5 — `PrestamoRepository`

In [14]:
class PrestamoRepository(RepositorioBase):
    """Persistencia de préstamos en prestamos.txt."""

    def __init__(self, ruta=None):
        """Usa por defecto RUTA_PRESTAMOS."""
        super().__init__(ruta or RUTA_PRESTAMOS, Prestamo)

    def find_by_codigo(self, cod):
        """Busca un préstamo por código; None si no existe."""
        return next((p for p in self.find_all() if p.codigo == cod.strip().upper()), None)

    def find_by_usuario(self, doc):
        """Préstamos de un usuario."""
        return [p for p in self.find_all() if p.num_documento == doc.strip()]

    def find_by_libro(self, isbn):
        """Préstamos de un libro."""
        return [p for p in self.find_all() if p.isbn == isbn.strip()]

    def actualizar(self, p):
        """Actualiza un préstamo existente."""
        self._reemplazar(p, "codigo")

## Celda 2.6 — `MultaRepository`

In [15]:
class MultaRepository(RepositorioBase):
    """Persistencia de multas en multas.txt."""

    def __init__(self, prestamo_repo, ruta=None):
        """Recibe el PrestamoRepository para poder asociar multas con usuarios."""
        super().__init__(ruta or RUTA_MULTAS, Multa)
        self.prestamo_repo = prestamo_repo

    def find_by_codigo(self, cod):
        """Busca una multa por código; None si no existe."""
        return next((m for m in self.find_all() if m.codigo == cod.strip().upper()), None)

    def find_by_prestamo(self, cod):
        """Busca la multa generada por un préstamo; None si no existe."""
        return next((m for m in self.find_all() if m.cod_prestamo == cod), None)

    def find_pendientes_usuario(self, doc):
        """Multas PENDIENTES de un usuario (a través de sus préstamos)."""
        codigos = {p.codigo for p in self.prestamo_repo.find_by_usuario(doc)}
        return [m for m in self.find_all()
                if m.cod_prestamo in codigos and m.estado == EstadoMultaEnum.PENDIENTE]

    def actualizar_estado(self, cod, estado):
        """Cambia el estado de una multa (recibe el texto del estado)."""
        m = self.find_by_codigo(cod)
        if m:
            m.estado = EstadoMultaEnum(estado)
            self._reemplazar(m, "codigo")

## Celda 2.7 — `ReservaRepository`

In [16]:
class ReservaRepository(RepositorioBase):
    """Persistencia de reservas en reservas.txt."""

    def __init__(self, ruta=None):
        """Usa por defecto RUTA_RESERVAS."""
        super().__init__(ruta or RUTA_RESERVAS, Reserva)

    def find_by_codigo(self, cod):
        """Busca una reserva por código; None si no existe."""
        return next((r for r in self.find_all() if r.codigo == cod.strip().upper()), None)

    def find_by_libro(self, isbn):
        """Reservas de un libro en orden de llegada (orden de registro en el archivo)."""
        return [r for r in self.find_all() if r.isbn == isbn.strip()]

    def find_by_usuario(self, doc):
        """Reservas de un usuario."""
        return [r for r in self.find_all() if r.num_documento == doc.strip()]

    def actualizar_estado(self, cod, estado):
        """Cambia el estado de una reserva (recibe el texto del estado)."""
        r = self.find_by_codigo(cod)
        if r:
            r.estado = EstadoReservaEnum(estado)
            self._reemplazar(r, "codigo")

# SECCIÓN 3 — SERVICIOS (lógica de negocio)
## Celda 3.1 — `UsuarioService`

In [17]:
class UsuarioService:
    """Reglas de negocio para la gestión de usuarios."""

    def __init__(self, usuario_repo):
        """Recibe el repositorio de usuarios."""
        self.repo = usuario_repo

    def registrar(self, num_documento, nombre, tipo, correo, telefono):
        """Registra un usuario nuevo validando campos obligatorios y documento único."""
        if not num_documento.strip() or not nombre.strip():
            raise ValueError("El documento y el nombre son obligatorios.")
        if "@" not in correo:
            raise ValueError("El correo no es válido.")
        if self.repo.find_by_documento(num_documento):
            raise ValueError(f"Ya existe un usuario con documento {num_documento}.")
        u = Usuario(num_documento, nombre, TipoUsuarioEnum(tipo), correo, telefono)
        self.repo.guardar(u)
        return u

    def buscar_por_documento(self, doc):
        """Devuelve el usuario con ese documento o lanza ValueError si no existe."""
        u = self.repo.find_by_documento(doc)
        if not u:
            raise ValueError(f"No existe un usuario con documento {doc}.")
        return u

    def actualizar(self, doc, correo=None, telefono=None, estado=None):
        """Actualiza correo, teléfono y/o estado (los valores vacíos no se modifican)."""
        u = self.buscar_por_documento(doc)
        if correo and correo.strip():
            if "@" not in correo:
                raise ValueError("El correo no es válido.")
            u.correo = correo.strip()
        if telefono and telefono.strip():
            u.telefono = telefono.strip()
        if estado:
            u.estado = estado
        self.repo.actualizar(u)
        return u

    def listar_todos(self):
        """Lista todos los usuarios registrados."""
        return self.repo.find_all()

## Celda 3.2 — `LibroService`

In [18]:
class LibroService:
    """Reglas de negocio para el catálogo de libros y categorías."""

    def __init__(self, libro_repo, categoria_repo):
        """Recibe los repositorios de libros y categorías."""
        self.libro_repo = libro_repo
        self.categoria_repo = categoria_repo

    def registrar_categoria(self, codigo, nombre, descripcion=""):
        """Registra una categoría nueva con código único."""
        if not codigo.strip() or not nombre.strip():
            raise ValueError("El código y el nombre de la categoría son obligatorios.")
        if self.categoria_repo.find_by_codigo(codigo):
            raise ValueError(f"Ya existe la categoría {codigo.upper()}.")
        c = Categoria(codigo, nombre, descripcion)
        self.categoria_repo.guardar(c)
        return c

    def listar_categorias(self):
        """Lista todas las categorías."""
        return self.categoria_repo.find_all()

    def registrar(self, isbn, titulo, autor, cod_categoria, anio, ejemplares):
        """Registra un libro validando ISBN único, categoría existente y cantidades."""
        if not isbn.strip() or not titulo.strip() or not autor.strip():
            raise ValueError("ISBN, título y autor son obligatorios.")
        if not cod_categoria or not self.categoria_repo.find_by_codigo(cod_categoria):
            raise ValueError("Seleccione una categoría existente (regístrela primero si no está).")
        if self.libro_repo.find_by_isbn(isbn):
            raise ValueError(f"Ya existe un libro con ISBN {isbn}.")
        anio, ejemplares = int(anio), int(ejemplares)
        if not (1000 <= anio <= date.today().year):
            raise ValueError("El año de publicación no es válido.")
        if ejemplares < 1:
            raise ValueError("El número de ejemplares debe ser al menos 1.")
        l = Libro(isbn, titulo, autor, cod_categoria, anio, ejemplares)
        self.libro_repo.guardar(l)
        return l

    def buscar_por_isbn(self, isbn):
        """Busca un libro por ISBN (lista con 0 o 1 elementos)."""
        l = self.libro_repo.find_by_isbn(isbn)
        return [l] if l else []

    def buscar_por_titulo(self, txt):
        """Busca libros por coincidencia parcial de título."""
        return self.libro_repo.find_by_titulo(txt)

    def listar_por_categoria(self, cod):
        """Lista los libros de una categoría."""
        return self.libro_repo.find_by_categoria(cod)

    def listar_disponibles(self):
        """Lista los libros con al menos un ejemplar disponible."""
        return [l for l in self.libro_repo.find_all() if l.disponibles > 0]

    def listar_todos(self):
        """Lista todo el catálogo."""
        return self.libro_repo.find_all()

## Celda 3.3 — `MultaService` *(se define antes que PrestamoService, que lo usa)*

In [19]:
class MultaService:
    """Cálculo, pago y consulta de multas."""
    VALOR_DIA = 1000   # COP por día de retraso

    def __init__(self, multa_repo):
        """Recibe el repositorio de multas."""
        self.repo = multa_repo

    def calcular_multa(self, prestamo):
        """Si el préstamo se devolvió tarde, crea y guarda la multa; si no, devuelve None."""
        if prestamo.esta_pendiente():
            return None
        dias = (parse_fecha(prestamo.fecha_devolucion) - parse_fecha(prestamo.fecha_limite)).days
        if dias <= 0 or self.repo.find_by_prestamo(prestamo.codigo):
            return None
        m = Multa(ArchivoUtil.generar_codigo("MUL"), prestamo.codigo, dias, dias * self.VALOR_DIA)
        self.repo.guardar(m)
        return m

    def pagar(self, cod_multa):
        """Registra el pago de una multa pendiente."""
        m = self.repo.find_by_codigo(cod_multa)
        if not m:
            raise ValueError(f"No existe la multa {cod_multa}.")
        if m.estado == EstadoMultaEnum.PAGADA:
            raise ValueError("Esa multa ya fue pagada.")
        self.repo.actualizar_estado(m.codigo, EstadoMultaEnum.PAGADA.value)
        m.estado = EstadoMultaEnum.PAGADA
        return m

    def listar_pendientes_usuario(self, doc):
        """Multas pendientes de un usuario."""
        return self.repo.find_pendientes_usuario(doc)

## Celda 3.4 — `PrestamoService`

In [20]:
class PrestamoService:
    """Reglas de negocio de préstamos y devoluciones."""

    def __init__(self, prestamo_repo, libro_repo, usuario_repo, reserva_repo, multa_service):
        """Recibe los repositorios necesarios y el servicio de multas."""
        self.repo = prestamo_repo
        self.libro_repo = libro_repo
        self.usuario_repo = usuario_repo
        self.reserva_repo = reserva_repo
        self.multa_service = multa_service

    def prestar(self, num_documento, isbn, fecha_prestamo):
        """Registra un préstamo validando usuario activo, multas, cupo y disponibilidad (con cola de reservas)."""
        f = parse_fecha(fecha_prestamo)
        u = self.usuario_repo.find_by_documento(num_documento)
        if not u:
            raise ValueError("El usuario no existe.")
        if not u.esta_activo():
            raise ValueError("El usuario está suspendido.")
        if self.multa_service.listar_pendientes_usuario(u.num_documento):
            raise ValueError("El usuario tiene multas pendientes.")
        pendientes = [p for p in self.repo.find_by_usuario(u.num_documento) if p.esta_pendiente()]
        if len(pendientes) >= u.tipo.cupo:
            raise ValueError(f"El usuario alcanzó su cupo ({u.tipo.cupo} libros para {u.tipo.value.lower()}).")
        if any(p.isbn == isbn.strip() for p in pendientes):
            raise ValueError("El usuario ya tiene prestado este libro.")
        libro = self.libro_repo.find_by_isbn(isbn)
        if not libro:
            raise ValueError("El libro no existe.")
        if libro.disponibles <= 0:
            raise ValueError("No hay ejemplares disponibles. Puede crear una reserva.")

        # Orden de llegada: los ejemplares libres se asignan primero a quienes reservaron antes
        cola = [r for r in self.reserva_repo.find_by_libro(libro.isbn) if r.estado == EstadoReservaEnum.ACTIVA]
        reserva_usuario = next((r for r in cola if r.num_documento == u.num_documento), None)
        if reserva_usuario:
            if cola.index(reserva_usuario) >= libro.disponibles:
                raise ValueError("Aún no es su turno en la cola de reservas de este libro.")
        elif len(cola) >= libro.disponibles:
            raise ValueError("Los ejemplares libres están reservados por otros usuarios. Puede reservar.")

        limite = f + timedelta(days=u.tipo.dias_prestamo)
        p = Prestamo(ArchivoUtil.generar_codigo("PRE"), libro.isbn, u.num_documento,
                     fmt_fecha(f), fmt_fecha(limite))
        self.repo.guardar(p)
        libro.disponibles -= 1
        self.libro_repo.actualizar(libro)
        if reserva_usuario:
            self.reserva_repo.actualizar_estado(reserva_usuario.codigo, EstadoReservaEnum.CUMPLIDA.value)
        return p

    def devolver(self, cod_prestamo, fecha_devolucion):
        """Registra la devolución, repone el ejemplar y genera la multa si hubo retraso. Devuelve (prestamo, multa|None)."""
        p = self.buscar_por_codigo(cod_prestamo)
        if not p.esta_pendiente():
            raise ValueError("Ese préstamo ya fue devuelto.")
        f = parse_fecha(fecha_devolucion)
        if f < parse_fecha(p.fecha_prestamo):
            raise ValueError("La fecha de devolución no puede ser anterior a la del préstamo.")
        p.fecha_devolucion = fmt_fecha(f)
        p.estado = EstadoPrestamoEnum.DEVUELTO
        self.repo.actualizar(p)
        libro = self.libro_repo.find_by_isbn(p.isbn)
        if libro:
            libro.disponibles = min(libro.disponibles + 1, libro.total_ejemplares)
            self.libro_repo.actualizar(libro)
        multa = self.multa_service.calcular_multa(p)
        return p, multa

    def buscar_por_codigo(self, cod):
        """Devuelve el préstamo con ese código o lanza ValueError."""
        p = self.repo.find_by_codigo(cod)
        if not p:
            raise ValueError(f"No existe el préstamo {cod}.")
        return p

    def historial_usuario(self, doc, solo_pendientes=False):
        """Préstamos de un usuario en orden cronológico."""
        lista = sorted(self.repo.find_by_usuario(doc), key=lambda p: p.fecha_prestamo)
        return [p for p in lista if p.esta_pendiente()] if solo_pendientes else lista

    def listar_vencidos(self, fecha):
        """Préstamos no devueltos cuya fecha límite es anterior a la fecha dada (y los marca VENCIDO)."""
        f = parse_fecha(fecha)
        vencidos = []
        for p in self.repo.find_all():
            if p.esta_pendiente() and parse_fecha(p.fecha_limite) < f:
                if p.estado != EstadoPrestamoEnum.VENCIDO:
                    p.estado = EstadoPrestamoEnum.VENCIDO
                    self.repo.actualizar(p)
                vencidos.append(p)
        return vencidos

## Celda 3.5 — `ReservaService`

In [21]:
class ReservaService:
    """Reglas de negocio de reservas (cola por orden de llegada)."""

    def __init__(self, reserva_repo, libro_repo, usuario_repo):
        """Recibe los repositorios de reservas, libros y usuarios."""
        self.repo = reserva_repo
        self.libro_repo = libro_repo
        self.usuario_repo = usuario_repo

    def reservar(self, num_documento, isbn, fecha):
        """Crea una reserva si el libro no tiene ejemplares libres sin comprometer."""
        f = parse_fecha(fecha)
        u = self.usuario_repo.find_by_documento(num_documento)
        if not u:
            raise ValueError("El usuario no existe.")
        if not u.esta_activo():
            raise ValueError("El usuario está suspendido.")
        libro = self.libro_repo.find_by_isbn(isbn)
        if not libro:
            raise ValueError("El libro no existe.")
        activas = self.listar_por_libro(libro.isbn)
        if any(r.num_documento == u.num_documento for r in activas):
            raise ValueError("El usuario ya tiene una reserva activa para este libro.")
        if libro.disponibles - len(activas) > 0:
            raise ValueError("Hay ejemplares disponibles: realice el préstamo directamente.")
        r = Reserva(ArchivoUtil.generar_codigo("RES"), libro.isbn, u.num_documento, fmt_fecha(f))
        self.repo.guardar(r)
        return r

    def cancelar(self, cod_reserva):
        """Cancela una reserva activa."""
        r = self.repo.find_by_codigo(cod_reserva)
        if not r:
            raise ValueError(f"No existe la reserva {cod_reserva}.")
        if r.estado != EstadoReservaEnum.ACTIVA:
            raise ValueError("Solo se pueden cancelar reservas activas.")
        self.repo.actualizar_estado(r.codigo, EstadoReservaEnum.CANCELADA.value)
        r.estado = EstadoReservaEnum.CANCELADA
        return r

    def listar_por_libro(self, isbn):
        """Reservas activas de un libro, en orden de llegada."""
        return [r for r in self.repo.find_by_libro(isbn) if r.estado == EstadoReservaEnum.ACTIVA]

    def listar_por_usuario(self, doc):
        """Todas las reservas de un usuario."""
        return self.repo.find_by_usuario(doc)

## Celda 3.6 — `AnalisisDatosService`

In [22]:
from collections import Counter


class AnalisisDatosService:
    """Indicadores estadísticos de uso de la colección."""

    def __init__(self, prestamo_repo, libro_repo, categoria_repo, usuario_repo, multa_repo):
        """Recibe los repositorios que alimentan los análisis."""
        self.prestamo_repo = prestamo_repo
        self.libro_repo = libro_repo
        self.categoria_repo = categoria_repo
        self.usuario_repo = usuario_repo
        self.multa_repo = multa_repo

    def libros_mas_prestados(self, desde, hasta, top=10):
        """Lista de (isbn, título, cantidad) de los libros más prestados entre dos fechas."""
        d, h = parse_fecha(desde), parse_fecha(hasta)
        cuenta = Counter(p.isbn for p in self.prestamo_repo.find_all()
                         if d <= parse_fecha(p.fecha_prestamo) <= h)
        resultado = []
        for isbn, n in cuenta.most_common(int(top)):
            libro = self.libro_repo.find_by_isbn(isbn)
            resultado.append((isbn, libro.titulo if libro else "(eliminado)", n))
        return resultado

    def prestamos_por_categoria(self):
        """Lista de (categoría, cantidad, porcentaje) de préstamos por categoría."""
        libros = {l.isbn: l.cod_categoria for l in self.libro_repo.find_all()}
        nombres = {c.codigo: c.nombre for c in self.categoria_repo.find_all()}
        cuenta = Counter(libros.get(p.isbn, "?") for p in self.prestamo_repo.find_all())
        total = sum(cuenta.values())
        return [(nombres.get(cod, cod), n, round(100 * n / total, 1))
                for cod, n in cuenta.most_common()] if total else []

    def morosidad_por_tipo(self, fecha_corte=None):
        """Lista de (tipo, préstamos, morosos, tasa%) — moroso: devuelto tarde o vencido sin devolver."""
        corte = parse_fecha(fecha_corte) if fecha_corte else date.today()
        tipos = {u.num_documento: u.tipo for u in self.usuario_repo.find_all()}
        total, morosos = Counter(), Counter()
        for p in self.prestamo_repo.find_all():
            tipo = tipos.get(p.num_documento)
            if tipo is None:
                continue
            total[tipo] += 1
            limite = parse_fecha(p.fecha_limite)
            tarde = (parse_fecha(p.fecha_devolucion) > limite) if not p.esta_pendiente() else (corte > limite)
            if tarde:
                morosos[tipo] += 1
        return [(t.value, total[t], morosos[t], round(100 * morosos[t] / total[t], 1) if total[t] else 0.0)
                for t in TipoUsuarioEnum]

    def duracion_promedio_prestamo(self):
        """Duración promedio real (días) de los préstamos devueltos; None si no hay datos."""
        dias = [(parse_fecha(p.fecha_devolucion) - parse_fecha(p.fecha_prestamo)).days
                for p in self.prestamo_repo.find_all() if not p.esta_pendiente()]
        return round(sum(dias) / len(dias), 2) if dias else None

    def reporte_multas(self):
        """Diccionario con cantidad de multas y valores generado, pagado y pendiente (COP)."""
        multas = self.multa_repo.find_all()
        pagado = sum(m.valor for m in multas if m.estado == EstadoMultaEnum.PAGADA)
        pendiente = sum(m.valor for m in multas if m.estado == EstadoMultaEnum.PENDIENTE)
        return {"cantidad": len(multas), "generado": pagado + pendiente,
                "pagado": pagado, "pendiente": pendiente}

## Celda 3.7 — Ensamblaje de dependencias (repositorios + servicios)

In [23]:
# Repositorios
usuario_repo   = UsuarioRepository()
categoria_repo = CategoriaRepository()
libro_repo     = LibroRepository()
prestamo_repo  = PrestamoRepository()
multa_repo     = MultaRepository(prestamo_repo)
reserva_repo   = ReservaRepository()

# Servicios
usuario_service  = UsuarioService(usuario_repo)
libro_service    = LibroService(libro_repo, categoria_repo)
multa_service    = MultaService(multa_repo)
prestamo_service = PrestamoService(prestamo_repo, libro_repo, usuario_repo, reserva_repo, multa_service)
reserva_service  = ReservaService(reserva_repo, libro_repo, usuario_repo)
analisis_service = AnalisisDatosService(prestamo_repo, libro_repo, categoria_repo, usuario_repo, multa_repo)
print("✅ Capas de modelo, repositorios y servicios listas.")

✅ Capas de modelo, repositorios y servicios listas.


# SECCIÓN 4 — INTERFAZ DE USUARIO (Gradio)
En esta versión los menús de `print()`/`input()` se reemplazan por una interfaz web con pestañas (una por módulo).
## Celda 4.0 — Funciones auxiliares de la interfaz

In [24]:
# --- SECCIÓN 4: INTERFAZ DE USUARIO ---------
# Celda 4.0 Funciones auxiliares de consola
from datetime import datetime, date
from IPython.display import clear_output

def limpiar_pantalla():
    """Limpia la salida de la celda en Google Colab."""
    clear_output(wait=True)

def leer_texto(mensaje):
    """Obliga al usuario a ingresar texto no vacío."""
    while True:
        txt = input(f"{mensaje}: ").strip()
        if txt:
            return txt
        print("❌ El valor no puede estar vacío.")

def leer_entero(mensaje):
    """Obliga al usuario a ingresar un número entero."""
    while True:
        try:
            return int(input(f"{mensaje}: ").strip())
        except ValueError:
            print("❌ Debe ingresar un número válido.")

def leer_fecha(mensaje):
    """Lee una fecha con formato o asume hoy si se deja en blanco."""
    while True:
        txt = input(f"{mensaje} (AAAA-MM-DD) [Enter para hoy]: ").strip()
        if not txt:
            return date.today().strftime('%Y-%m-%d')
        try:
            datetime.strptime(txt, '%Y-%m-%d')
            return txt
        except ValueError:
            print("❌ Formato inválido. Use AAAA-MM-DD.")

## Celda 4.1 — Módulo Usuarios

In [25]:
# Celda 4.1 menu_usuarios()
def menu_usuarios():
    while True:
        print("\n--- 👥 MÓDULO DE USUARIOS ---")
        print("1. Registrar usuario")
        print("2. Consultar / Actualizar usuario")
        print("3. Listar todos los usuarios")
        print("0. Volver al menú principal")
        op = input("Seleccione una opción: ").strip()

        if op == '1':
            doc = leer_texto("Documento")
            nom = leer_texto("Nombre completo")
            print("Tipos: 1. ESTUDIANTE | 2. DOCENTE | 3. EXTERNO")
            tipo_idx = input("Seleccione tipo (1/2/3): ").strip()
            tipo = {"1": "ESTUDIANTE", "2": "DOCENTE", "3": "EXTERNO"}.get(tipo_idx, "ESTUDIANTE")
            correo = leer_texto("Correo")
            tel = leer_texto("Teléfono")
            try:
                u = usuario_service.registrar(doc, nom, tipo, correo, tel)
                print(f"✅ Usuario {u.nombre} registrado con éxito.")
            except ValueError as e:
                print(f"❌ Error: {e}")

        elif op == '2':
            doc = leer_texto("Ingrese documento a consultar")
            try:
                u = usuario_service.buscar_por_documento(doc)
                print(f"\nDatos: {u.nombre} | {u.tipo.value} | {u.correo} | Tel: {u.telefono} | {u.estado}")
                print("--- Actualización (Deje en blanco para mantener igual) ---")
                correo = input("Nuevo correo: ").strip()
                tel = input("Nuevo teléfono: ").strip()
                est = input("Nuevo estado (ACTIVO/SUSPENDIDO): ").strip().upper()

                est_final = est if est in ["ACTIVO", "SUSPENDIDO"] else None
                u_act = usuario_service.actualizar(doc, correo, tel, est_final)
                print(f"✅ Usuario {u_act.nombre} actualizado.")
            except ValueError as e:
                print(f"❌ Error: {e}")

        elif op == '3':
            usuarios = usuario_service.listar_todos()
            print("\n--- LISTA DE USUARIOS ---")
            for u in usuarios:
                print(f"{u.num_documento} | {u.nombre} | {u.tipo.value} | {u.estado}")

        elif op == '0':
            break
        else:
            print("❌ Opción inválida.")

## Celda 4.2 — Módulo Libros y categorías

In [26]:
# Celda 4.2 menu_libros()
def menu_libros():
    while True:
        print("\n--- 📚 MÓDULO DE CATÁLOGO ---")
        print("1. Registrar categoría")
        print("2. Registrar libro")
        print("3. Consultar libros")
        print("0. Volver")
        op = input("Seleccione una opción: ").strip()

        if op == '1':
            cod = leer_texto("Código de categoría (Ej. LIT)").upper()
            nom = leer_texto("Nombre")
            desc = input("Descripción: ").strip()
            try:
                c = libro_service.registrar_categoria(cod, nom, desc)
                print(f"✅ Categoría {c.nombre} guardada.")
            except ValueError as e:
                print(f"❌ Error: {e}")

        elif op == '2':
            isbn = leer_texto("ISBN")
            tit = leer_texto("Título")
            aut = leer_texto("Autor")
            cats = libro_service.listar_categorias()
            print("Categorías:", ", ".join([c.codigo for c in cats]))
            cat = leer_texto("Código de categoría seleccionada").upper()
            anio = leer_entero("Año de publicación")
            ejem = leer_entero("Total de ejemplares")
            try:
                l = libro_service.registrar(isbn, tit, aut, cat, anio, ejem)
                print(f"✅ Libro '{l.titulo}' guardado.")
            except ValueError as e:
                print(f"❌ Error: {e}")

        elif op == '3':
            print("1. Por ISBN | 2. Por Título | 3. Listar Disponibles")
            sub = input("Buscar por: ").strip()
            try:
                if sub == '1':
                    libros = libro_service.buscar_por_isbn(leer_texto("ISBN"))
                elif sub == '2':
                    libros = libro_service.buscar_por_titulo(leer_texto("Título"))
                elif sub == '3':
                    libros = libro_service.listar_disponibles()
                else:
                    print("❌ Opción inválida.")
                    continue

                print("\n--- RESULTADOS ---")
                if not libros: print("No se encontraron coincidencias.")
                for l in libros:
                    print(f"{l.isbn} | {l.titulo} | {l.autor} | Disp: {l.disponibles}/{l.total_ejemplares}")
            except ValueError as e:
                print(f"❌ Error: {e}")

        elif op == '0':
            break

## Celda 4.3 — Módulo Préstamos

In [27]:
# Celda 4.3 menu_prestamos()
def menu_prestamos():
    while True:
        print("\n--- 🔁 MÓDULO DE PRÉSTAMOS ---")
        print("1. Registrar préstamo")
        print("2. Registrar devolución")
        print("3. Historial de usuario")
        print("4. Listar préstamos vencidos")
        print("0. Volver")
        op = input("Seleccione una opción: ").strip()

        if op == '1':
            doc = leer_texto("Documento del usuario")
            isbn = leer_texto("ISBN del libro")
            fecha = leer_fecha("Fecha del préstamo")
            try:
                p = prestamo_service.prestar(doc, isbn, fecha)
                print(f"✅ Préstamo {p.codigo} exitoso. Fecha límite: {p.fecha_limite}")
            except ValueError as e:
                print(f"❌ Error: {e}")

        elif op == '2':
            cod = leer_texto("Código del préstamo (PRE-... )").upper()
            fecha = leer_fecha("Fecha de devolución real")
            try:
                p, m = prestamo_service.devolver(cod, fecha)
                print(f"✅ Devolución procesada (Préstamo: {p.codigo}).")
                if m:
                    print(f"⚠️ ATENCIÓN: Retraso de {m.dias_retraso} días. Se generó multa {m.codigo} por ${m.valor} COP.")
            except ValueError as e:
                print(f"❌ Error: {e}")

        elif op == '3':
            doc = leer_texto("Documento del usuario")
            try:
                historial = prestamo_service.historial_usuario(doc)
                print(f"\n--- HISTORIAL DE {doc} ---")
                for p in historial:
                    print(f"{p.codigo} | Libro: {p.isbn} | {p.fecha_prestamo} a {p.fecha_limite} | Est: {p.estado.value}")
            except ValueError as e:
                print(f"❌ Error: {e}")

        elif op == '4':
            fecha = leer_fecha("Consultar vencidos a la fecha")
            vencidos = prestamo_service.listar_vencidos(fecha)
            print(f"\n--- VENCIDOS AL {fecha} ---")
            for p in vencidos:
                print(f"{p.codigo} | Usr: {p.num_documento} | Libro: {p.isbn} | Límite: {p.fecha_limite}")

        elif op == '0':
            break

## Celda 4.4 — Módulo Reservas y multas

In [28]:
# Celda 4.4 menu_reservas_multas()
def menu_reservas_multas():
    while True:
        print("\n--- 📋 RESERVAS Y MULTAS ---")
        print("1. Crear reserva")
        print("2. Cancelar reserva")
        print("3. Pagar multa")
        print("4. Ver multas pendientes por usuario")
        print("0. Volver")
        op = input("Seleccione una opción: ").strip()

        if op == '1':
            doc = leer_texto("Documento del usuario")
            isbn = leer_texto("ISBN del libro")
            fecha = leer_fecha("Fecha de reserva")
            try:
                r = reserva_service.reservar(doc, isbn, fecha)
                print(f"✅ Reserva {r.codigo} creada correctamente.")
            except ValueError as e:
                print(f"❌ Error: {e}")

        elif op == '2':
            cod = leer_texto("Código de reserva (RES-... )").upper()
            try:
                r = reserva_service.cancelar(cod)
                print(f"✅ Reserva {r.codigo} cancelada.")
            except ValueError as e:
                print(f"❌ Error: {e}")

        elif op == '3':
            cod = leer_texto("Código de multa (MUL-... )").upper()
            try:
                m = multa_service.pagar(cod)
                print(f"✅ Multa {m.codigo} pagada (${m.valor} COP).")
            except ValueError as e:
                print(f"❌ Error: {e}")

        elif op == '4':
            doc = leer_texto("Documento del usuario")
            try:
                pendientes = multa_service.listar_pendientes_usuario(doc)
                print(f"\n--- MULTAS PENDIENTES ---")
                total = 0
                for m in pendientes:
                    print(f"{m.codigo} | Préstamo: {m.cod_prestamo} | Retraso: {m.dias_retraso} días | Valor: ${m.valor}")
                    total += m.valor
                print(f"💰 Deuda Total: ${total} COP")
            except ValueError as e:
                print(f"❌ Error: {e}")

        elif op == '0':
            break

## Celda 4.5 — Módulo Análisis de datos

In [29]:
# Celda 4.5 menu_analisis()
def menu_analisis():
    while True:
        print("\n--- 📊 ANÁLISIS DE DATOS ---")
        print("1. Listar los N libros más prestados en un periodo")
        print("2. Ver distribución de préstamos por categoría")
        print("3. Ver tasa de morosidad por tipo de usuario")
        print("4. Ver duración promedio de los préstamos devueltos")
        print("5. Ver reporte de multas")
        print("0. Volver")
        op = input("Seleccione una opción: ").strip()

        if op == '1':
            ini = leer_fecha("Fecha inicio")
            fin = leer_fecha("Fecha fin")
            top = leer_entero("Cantidad (N) de libros a mostrar")
            stats = analisis_service.libros_mas_prestados(ini, fin, top)
            print(f"\n--- TOP {top} MÁS PRESTADOS ({ini} a {fin}) ---")
            for isbn, tit, cant in stats:
                print(f"{isbn} | {tit} | {cant} préstamo(s)")

        elif op == '2':
            stats = analisis_service.prestamos_por_categoria()
            print("\n--- DISTRIBUCIÓN POR CATEGORÍA ---")
            for cat, cant, pct in stats:
                print(f"{cat}: {cant} préstamos ({pct}%)")

        elif op == '3':
            corte = leer_fecha("Fecha de corte")
            stats = analisis_service.morosidad_por_tipo(corte)
            print("\n--- TASA DE MOROSIDAD ---")
            print(f"{'Tipo':<12} | {'Total':<10} | {'Morosos':<10} | {'Tasa %'}")
            print("-" * 50)
            for tipo, tot, mor, tasa in stats:
                print(f"{tipo:<12} | {tot:<10} | {mor:<10} | {tasa}%")

        elif op == '4':
            promedio = analisis_service.duracion_promedio_prestamo()
            if promedio is None:
                print("\nNo hay préstamos devueltos aún.")
            else:
                print(f"\n⏱️ Duración promedio de retención de libros: {promedio} días.")

        elif op == '5':
            rep = analisis_service.reporte_multas()
            print("\n--- REPORTE FINANCIERO DE MULTAS ---")
            print(f"Total multas generadas (Cantidad): {rep['cantidad']}")
            print(f"Valor Generado Histórico: ${rep['generado']:,} COP")
            print(f"Valor Pagado (Ingresos):  ${rep['pagado']:,} COP")
            print(f"Valor Pendiente (Deuda):  ${rep['pendiente']:,} COP")

        elif op == '0':
            break

## Celda 4.6 — Interfaz principal (equivale a `menu_principal()`)

In [30]:
# Celda 4.6 menu_principal()
def menu_principal():
    while True:
        limpiar_pantalla()
        print("="*45)
        print(" 🏛️ SISTEMA DE GESTIÓN BIBLIOTECARIA (SGB)")
        print("="*45)
        print("1. Gestión de Usuarios")
        print("2. Catálogo de Libros")
        print("3. Préstamos y Devoluciones")
        print("4. Reservas y Multas")
        print("5. Análisis de Datos")
        print("0. Salir del SGB")

        opcion = input("Seleccione una opción (0-5): ").strip()

        if opcion == '1':
            limpiar_pantalla()
            menu_usuarios()
        elif opcion == '2':
            limpiar_pantalla()
            menu_libros()
        elif opcion == '3':
            limpiar_pantalla()
            menu_prestamos()
        elif opcion == '4':
            limpiar_pantalla()
            menu_reservas_multas()
        elif opcion == '5':
            limpiar_pantalla()
            menu_analisis()
        elif opcion == '0':
            print("👋 Cerrando SGB... ¡Hasta pronto!")
            break
        else:
            print("❌ Opción inválida.")

# SECCIÓN 5 — EJECUCIÓN PRINCIPAL
## Celda 5.1 — Iniciar el sistema

In [31]:
# --- SECCIÓN 5: EJECUCIÓN ---------
# Celda 5.1 menu_principal() - ejecutar para iniciar

menu_principal()


--- 👥 MÓDULO DE USUARIOS ---
1. Registrar usuario
2. Consultar / Actualizar usuario
3. Listar todos los usuarios
0. Volver al menú principal


KeyboardInterrupt: Interrupted by user